In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"]=""

In [ ]:
import sys
!{sys.executable} -m pip install -q youtube-transcript-api langchain-community langchain_huggingface faiss-cpu tiktoken python-dotenv yt-dlp


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
import subprocess
import re

Step-1a-Indexing (Document Ingestion)

In [ ]:

video_id = "Gfr50f6ZBvo"
url = f"https://www.youtube.com/watch?v={video_id}"

# This saves English auto-generated subs into "output.en.srt"
subprocess.run([
    "yt-dlp", "--skip-download",
    "--write-auto-subs", "--sub-lang", "en",
    "--convert-subs", "srt",
    "-o", "output", url
])


In [ ]:
def clean_vtt(vtt_file):
    with open(vtt_file, "r", encoding="utf-8") as f:
        text = f.read()

    # Remove WEBVTT header & metadata
    text = re.sub(r"WEBVTT.*\n", "", text)
    text = re.sub(r"Kind:.*\n", "", text)
    text = re.sub(r"Language:.*\n", "", text)

    # Remove timestamps like 00:00:01.200 --> 00:00:04.500
    text = re.sub(r"\d{2}:\d{2}:\d{2}\.\d{3} --> .*", "", text)

    # Remove inline word-level timestamps <00:00:00.160>
    text = re.sub(r"<\d{2}:\d{2}:\d{2}\.\d{3}>", "", text)

    # Remove <c> and </c> tags
    text = re.sub(r"</?c>", "", text)

    # Remove extra spaces & newlines
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return " ".join(lines)
     

transcript_clean = clean_vtt("output.en.vtt")
print(transcript_clean)  # preview first 1000 chars

Step 1b-Indexing (Text Splitting)

In [ ]:
splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks=splitter.create_documents([transcript_clean])

In [ ]:
len(chunks)

In [ ]:
chunks[0]

Step 1 & 1d - Indexing(Embedding Generation and Storing in Vector Store)

In [ ]:
embedding= HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vector_store=FAISS.from_documents(chunks,embedding)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['7e21c02a-3f09-4710-8c29-97a123972346'])

Step-2 Retrieval

In [ ]:
retriever=vector_store.as_retriever(search_type="similarity", search_kwargs={"k":4})

In [ ]:
retriever

In [ ]:
retriever.invoke("what is deepmind")

Step-3 Augmentation

In [ ]:
prompt=PromptTemplate(
    template="""
    You are a helpful assistance.
    Answer ONLY from the provided transcript context.
    If the context is insufficient, just say you don't know.

    {context}
    Question: {question}
""",
input_variables=['context','question']
)

In [ ]:
question="Is the topic of aliens discussed in the video? If yes then what was discussed?"
retrieved_docs=retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

In [ ]:
final_prompt=prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

Step-4 - Generation

In [ ]:
llm= HuggingFaceEndpoint(
    repo_id="google/gemma-2-2b-it",
    task="text-generation"
)
model=ChatHuggingFace(llm=llm)

In [ ]:
answer=model.invoke(final_prompt)
print(answer.content)

Building a Chain

In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(Retrieved_docs):
    context_text="\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [ ]:
parallel_chain=RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser=StrOutputParser()

In [ ]:
main_chain=parallel_chain | prompt | model | parser

In [ ]:
main_chain.invoke("Can you summerize the video")